In [ ]:
import json
from datasets import load_dataset

with open("word_to_idx.json", "r") as f:
    word_to_idx = json.load(f)
with open("idx_to_word.json", "r") as f:
    idx_to_word = json.load(f)

with open("inverse_index.json", "r") as f:
    inverse_index = json.load(f)

with open("problem_to_keywords.json", "r") as f:
    problem_to_keywords = json.load(f)

dump = load_dataset("heegyu/namuwiki-sentences", split="train", streaming=True)

In [ ]:
from math import log2

score_of_keyword = {}

for idx_keyword in range(len(idx_to_word)):
    set_sentence = set()
    set_sentence.update(inverse_index[str(idx_keyword)])

    set_title = set()
    set_title.update([dump[idx_sentence]['title'] for idx_sentence in set_sentence])

    if 0 < len(set_title) < 32768:
        score_of_keyword[str(idx_keyword)] = 1.0 + log2(32768) - log2(len(set_title))

with open("score_of_keyword.json", "w") as f:
    json.dump(score_of_keyword, f, indent=4)

In [ ]:
problem_to_inverse_index = [[] for _ in range(869)]
for problem, keywords in problem_to_keywords.items():
    tmp = set()
    for i in keywords:
        if str(i) in score_of_keyword:
            tmp.update(inverse_index[str(i)])
    problem_to_inverse_index[int(problem)] = sorted(tmp)

In [ ]:
def BM25(idx_problem):
    ret = []
    if not problem_to_inverse_index[idx_problem]:
        return ret
    
    list_keyword = problem_to_keywords[str(idx_problem)]

    for idx_sentence in problem_to_inverse_index[idx_problem]:
        score_of_sentence = 0.0
        for idx_keyword in list_keyword:
            if str(idx_keyword) not in score_of_keyword:
                continue

            count = inverse_index[str(idx_keyword)].count(idx_sentence)

            score_of_sentence += score_of_keyword[str(idx_keyword)] * count / (count + 0.25)

        if score_of_sentence > 3.0:
            ret.append((score_of_sentence, idx_sentence))
    return ret

In [ ]:
import pandas as pd
import ast

df_test = pd.read_csv("test.csv", index_col=0)
problem_to_RAGlist = {}

for idx_problem in range(869):
    row = df_test.iloc[idx_problem]
    problem = ast.literal_eval(row['problems'])
    user_content = f"<제시문>\n{row['paragraph']}\n\n"
    if pd.notna(row['question_plus']):
        user_content += f"<보기>\n{row['question_plus']}\n\n"
    user_content += f"<질문>\n{problem['question']}\n"
    for k in range(len(problem['choices'])):
        user_content += f"{k+1}. {problem['choices'][k]}\n"

    df_test.loc[idx_problem, 'user_content'] = user_content
    if len(user_content) > 900:
        continue

    bm25_result = BM25(idx_problem)
    if bm25_result:
        bm25_result.sort(reverse=True)
        refined_result = bm25_result[:min(512, len(bm25_result))]
        print(idx_problem, refined_result)
        problem_to_RAGlist[idx_problem] = [idx_sentence for _, idx_sentence in refined_result]

with open("problem_to_RAGlist.json", "w") as f:
    json.dump(problem_to_RAGlist, f, indent=4)

In [ ]:
import json
from datasets import load_dataset

with open("problem_to_RAGlist.json", "r") as f:
    problem_to_RAGlist = json.load(f)

dump = load_dataset("heegyu/namuwiki-sentences", split="train", streaming=False)

In [ ]:
for idx_problem, RAGlist in problem_to_RAGlist.items():
    print(idx_problem)
    for t, idx_sentence in enumerate(RAGlist):
        ran = (idx_sentence, idx_sentence + 1)
        length = len(dump[idx_sentence]["sentence"])
        while ran[1] - ran[0] < 40:
            conti = False
            if ran[0] > 0 and dump[ran[0] - 1]["title"] == dump[ran[0]]["title"] and dump[ran[0] - 1]["pi"] == dump[ran[0]]["pi"]:
                new_length = length + len(dump[ran[0] - 1]["sentence"])
                if new_length < 800:
                    ran = (ran[0] - 1, ran[1])
                    length = new_length
                    conti = True
            if ran[1] < len(dump) and dump[ran[1]]["title"] == dump[ran[1] - 1]["title"] and dump[ran[1]]["pi"] == dump[ran[1] - 1]["pi"]:
                new_length = length + len(dump[ran[1]]["sentence"])
                if new_length < 800:
                    ran = (ran[0], ran[1] + 1)
                    length = new_length
                    conti = True
            if not conti:
                break
        
        while ran[1] - ran[0] < 30:
            conti = False
            if ran[0] > 0 and dump[ran[0] - 1]["title"] == dump[ran[0]]["title"]:
                new_length = length + len(dump[ran[0] - 1]["sentence"])
                if new_length < 600:
                    ran = (ran[0] - 1, ran[1])
                    length = new_length
                    conti = True
            if ran[1] < len(dump) and dump[ran[1]]["title"] == dump[ran[1] - 1]["title"]:
                new_length = length + len(dump[ran[1]]["sentence"])
                if new_length < 600:
                    ran = (ran[0], ran[1] + 1)
                    length = new_length
                    conti = True
            if not conti:
                break
        
        text = ""
        titl = dump[idx_sentence]['title']
        if "대학수학능력시험" not in titl and "수능" not in titl and "탐구 영역" not in titl:
            text += f"<{titl}>\n"
            for i in range(ran[0], ran[1]):
                text += dump[i]["sentence"] + " "
        
        df_test.loc[int(idx_problem), f"RAG_{t}"] = text.strip()
df_test.to_csv("test_with_RAG512.csv", index=False)